# 01 · Source Profiling

Objetivo: iniciar el primer paso offline del capstone para perfilar las fuentes de pacientes en `Data_Source/1_Sources/`, revisar la estructura disponible y validar que los reportes derivados puedan consumirse desde el cuaderno.

Este cuaderno acompaña al script reusable `scripts/profile_sources.py`.

In [ ]:
from pathlib import Path

import pandas as pd

## Configurar datos o parámetros iniciales

Se definen las rutas base del workspace, la carpeta de fuentes y la carpeta de reportes derivados.

In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE_DIR = BASE_DIR / "Data_Source" / "1_Sources"
REPORT_DIR = BASE_DIR / "outputs" / "reports"

csv_files = sorted(SOURCE_DIR.glob("*.csv"))

BASE_DIR, SOURCE_DIR, REPORT_DIR, len(csv_files)

## Crear la primera celda de ejecución

Esta celda inspecciona los CSV fuente, muestra cantidad de filas y columnas por archivo, y verifica si ya existen los reportes generados por el script.

In [ ]:
source_overview = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file, dtype=str, keep_default_na=False)
    source_overview.append(
        {
            "file_name": csv_file.name,
            "rows": len(df),
            "columns": len(df.columns),
            "first_columns": ", ".join(df.columns[:5]),
        }
    )

source_overview_df = pd.DataFrame(source_overview)

summary_path = REPORT_DIR / "source_profile_summary.csv"
issues_path = REPORT_DIR / "patient_quality_issues.csv"

source_overview_df

## Verificar la salida inicial

Si los reportes ya existen, esta celda muestra un resumen por archivo y los tipos de issue más frecuentes. Si no existen, deja indicado el siguiente paso: ejecutar `scripts/profile_sources.py`.

In [ ]:
if summary_path.exists() and issues_path.exists():
    summary_df = pd.read_csv(summary_path)
    issues_df = pd.read_csv(issues_path)

    display(summary_df)
    display(
        issues_df.groupby(["file_name", "issue_type"], dropna=False)
        .size()
        .reset_index(name="issue_count")
        .sort_values(["file_name", "issue_count"], ascending=[True, False])
        .head(20)
    )
else:
    print("Aún no existen los reportes derivados.")
    print("Siguiente paso: ejecutar scripts/profile_sources.py para generar source_profile_summary.csv y patient_quality_issues.csv.")